In [ ]:
! pip install synthcity sdv

In [101]:
import pandas as pd
import numpy as np
import torch
import random
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from synthcity.plugins import Plugins
from sdv.metadata import SingleTableMetadata
import tqdm

In [73]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

In [109]:
dir_datasets = '/kaggle/input/real-syn-data/data/processed/'

# Загрузка реальных датасетов
real_data_1 = pd.read_csv(dir_datasets+'abolone.csv')
real_data_2 = pd.read_csv(dir_datasets+'insurance.csv')
real_data_3 = pd.read_csv(dir_datasets+'two_moons.csv')

# Словарь датасетов для удобства 
datasets = {
    # 'abolone': real_data_1,
    'insurance': real_data_2,
    #'two_moons': real_data_3
}

for name, data in datasets.items():
    print(f"{name}: {data.shape}, колонки: {list(data.columns)}")

insurance: (1338, 7), колонки: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'expenses']


In [75]:
from synthcity.plugins.generic.plugin_ddpm import TabDDPMPlugin as DDPM

In [76]:
# Определение пространства поиска гиперпараметров для DDPM
ddpm_space = {
    'batch_size': hp.choice('batch_size', [4096]),
    'lr': hp.qloguniform('lr', np.log(3.5e-4), np.log(9.2e-4), 1e-5),
    'num_timesteps': hp.choice('num_timesteps', [1000]),
    'n_layers_hidden': hp.choice('n_layers_hidden', [2, 4, 6]),
    'n_units_hidden': hp.choice('dim_hidden', [256, 512, 1024])
}

In [77]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

def evaluate_c2st(real, synthetic):
    real_copy = real.copy()

    if hasattr(synthetic, "dataframe"):
        synthetic = synthetic.dataframe()
        
    synthetic_copy = synthetic.copy()
    
    # Добавление меток
    real_copy['label'] = 1
    synthetic_copy['label'] = 0
    
    # Объединение данных
    df = pd.concat([real_copy, synthetic_copy], ignore_index=True)
    
    # Определение категориальных колонок
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
    categorical_columns = [col for col in categorical_columns if col != 'label']
    
    # Разделение на признаки и метки
    X = df.drop(columns='label')
    y = df['label']
    
    # Разделение на обучающую и тестовую выборки
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=RANDOM_SEED
    )
    
    # Обучение CatBoost
    clf = CatBoostClassifier(
        random_seed=RANDOM_SEED,
        verbose=0
    )
    
    clf.fit(X_train, y_train, cat_features=categorical_columns, eval_set=(X_test, y_test))
    
    # Предсказание и вычисление метрик
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    return f1 #accuracy

In [105]:
import sys
import os
import builtins
import contextlib

@contextlib.contextmanager
def suppress_all_output():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        old_print = builtins.print
        builtins.print = lambda *args, **kwargs: None
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr
            builtins.print = old_print

In [106]:
def generate_and_evaluate(params, dataset):
    #Извлечение метаданных
    metadata = SingleTableMetadata()
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    metadata.detect_from_dataframe(data=dataset)
    
    
    # Создание и обучение DDPM с заданными параметрами
    ddpm = DDPM(
        lr=params['lr'],
        num_timesteps=params['num_timesteps'],
        model_params = dict(n_layers_hidden=params['n_layers_hidden'], 
                            n_units_hidden=params['n_units_hidden'], 
                            dropout=0.0),
    )
    
    # Обучение модели на реальных данных
    with suppress_all_output():
        ddpm.fit(dataset)
    
    # Генерация синтетических данных
    synthetic_data = ddpm.generate(len(dataset))
    
    # Вычисление метрики C2ST
    c2st_score = evaluate_c2st(dataset, synthetic_data)

    return c2st_score

In [107]:
def optimize_dataset(dataset_name, dataset, max_evals=30):
    print(f"Оптимизация для датасета: {dataset_name}, размер: {dataset.shape}")
    
    # Создаем объект для хранения истории поиска
    trials = Trials()
    
    # Определяем целевую функцию для оптимизации
    def objective(params):
        print("="*50)
        c2st_score = generate_and_evaluate(params, dataset)

        print(c2st_score)
        print(params)
        print("="*50)
        # Возвращаем словарь в формате, который ожидает hyperopt
        return {
            'loss': c2st_score,  
            'status': STATUS_OK,
            'c2st_score': c2st_score  
        }
    
    
    # Запускаем оптимизацию с помощью TPE алгоритма
    rng = np.random.default_rng(RANDOM_SEED)
    best = fmin(
        fn=objective,
        space=ddpm_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=rng
    )
    
    # Получаем лучшие параметры в читаемом виде
    best_params = {
        'batch_size': 4096,
        'num_timesteps': 1000,
        'lr': [best['lr']],
        'n_layers_hidden': [2, 4, 6][best['n_layers_hidden']],
        'n_units_hidden': [256, 512, 1024][best['n_units_hidden']],
    }
    
    # Находим лучший результат
    best_trial = min(trials.trials, key=lambda x: x['result']['loss'])
    best_c2st = best_trial['result'].get('c2st_score', None)
    
    results = {
        'best_params': best_params,
        'best_c2st': best_c2st,
        'c2st_diff_from_optimal': best_trial['result']['loss']
    }
    
    print(f"Лучший C2ST для {dataset_name}: {best_c2st}")
    print(f"Лучшие параметры: {best_params}")
    print('-' * 50)
    
    return results

In [108]:
# Словарь для хранения результатов
all_results = {}

# Указываем количество итераций для каждого датасета
max_evals = 50 

# Запускаем оптимизацию для каждого датасета
for dataset_name in datasets:
    data = datasets[dataset_name]
    all_results[dataset_name] = optimize_dataset(dataset_name, data, max_evals)

Оптимизация для датасета: abolone, размер: (4177, 9)
0.830289193302892                                     
{'batch_size': 4096, 'lr': 0.00064, 'n_layers_hidden': 2, 'n_units_hidden': 1024, 'num_timesteps': 1000}
0.8386134923592993                                                                
{'batch_size': 4096, 'lr': 0.00063, 'n_layers_hidden': 6, 'n_units_hidden': 256, 'num_timesteps': 1000}
0.8314018691588785                                                                
{'batch_size': 4096, 'lr': 0.00038, 'n_layers_hidden': 2, 'n_units_hidden': 256, 'num_timesteps': 1000}
0.8358878504672898                                                                
{'batch_size': 4096, 'lr': 0.00043000000000000004, 'n_layers_hidden': 4, 'n_units_hidden': 512, 'num_timesteps': 1000}
0.8195836545875096                                                                
{'batch_size': 4096, 'lr': 0.00038, 'n_layers_hidden': 2, 'n_units_hidden': 512, 'num_timesteps': 1000}
1.0                     

KeyError: 'n_units_hidden'

### best loss: 0.8175238095238095                                                                  
### {'batch_size': 4096, 'lr': 0.00051, 'n_layers_hidden': 2, 'n_units_hidden': 512, 'num_timesteps': 1000}

In [111]:
# Словарь для хранения результатов
all_results = {}

# Указываем количество итераций для каждого датасета
max_evals = 50 

# Запускаем оптимизацию для каждого датасета
for dataset_name in datasets:
    data = datasets[dataset_name]
    all_results[dataset_name] = optimize_dataset(dataset_name, data, max_evals)

Оптимизация для датасета: insurance, размер: (1338, 7)
0.9196217494089834                                    
{'batch_size': 4096, 'lr': 0.00064, 'n_layers_hidden': 2, 'n_units_hidden': 1024, 'num_timesteps': 1000}
0.9376498800959233                                                              
{'batch_size': 4096, 'lr': 0.00063, 'n_layers_hidden': 6, 'n_units_hidden': 256, 'num_timesteps': 1000}
0.9387755102040817                                                              
{'batch_size': 4096, 'lr': 0.00038, 'n_layers_hidden': 2, 'n_units_hidden': 256, 'num_timesteps': 1000}
0.9346246973365617                                                              
{'batch_size': 4096, 'lr': 0.00043000000000000004, 'n_layers_hidden': 4, 'n_units_hidden': 512, 'num_timesteps': 1000}
0.9252669039145908                                                              
{'batch_size': 4096, 'lr': 0.00038, 'n_layers_hidden': 2, 'n_units_hidden': 512, 'num_timesteps': 1000}
0.9526123936816525            

KeyError: 'n_units_hidden'

### best loss: 0.9075829383886256                                                               
### {'batch_size': 4096, 'lr': 0.0007400000000000001, 'n_layers_hidden': 2, 'n_units_hidden': 1024, 'num_timesteps': 1000}